# The Solar Oscillations Investigation - Michelson Doppler Imager — Implementation / 구현

**Paper**: Scherrer, P. H., et al. (1995). "The Solar Oscillations Investigation - Michelson Doppler Imager." *Solar Physics* 162, 129-188. DOI: 10.1007/BF00733429

This notebook reproduces three core algorithms of MDI:

1. The **Lyot + dual-Michelson cascade transmission profile** that defines MDI's 94 mÅ FWHM passband.
2. The **MDI Doppler velocity proxy** $\alpha$ from five filtergrams across the Ni I 6768 Å line, including its lookup-table calibration curve.
3. A simulated **$k$-$\omega$ (or $\ell$-$\nu$) p-mode power spectrum** from a synthetic time series of solar oscillations.

이 노트북은 MDI의 세 가지 핵심 알고리즘을 재현합니다: (1) MDI의 94 mÅ FWHM 통과대를 결정하는 Lyot + 듀얼 Michelson 캐스케이드 투과 프로파일, (2) Ni I 6768 Å 선의 5개 필터그램으로부터 MDI 도플러 속도 프록시 $\alpha$ 및 lookup-table 보정 곡선, (3) 합성 태양 진동 시계열로부터 $k$-$\omega$ p-mode 파워 스펙트럼.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Physical constants and MDI parameters / 물리 상수 및 MDI 매개변수
C_LIGHT = 2.99792458e8  # m/s
LAMBDA_NI = 6767.78     # Angstrom; Ni I rest wavelength used by MDI
MICHELSON_M1_FSR = 377.0  # mAA; free spectral range of M1 (FWHM = 188 mAA)
MICHELSON_M2_FSR = 189.0  # mAA; free spectral range of M2 (FWHM = 94 mAA)
LYOT_FWHM = 465.0       # mAA; Lyot filter FWHM
TUNING_STEP = 75.0      # mAA; spacing between filtergrams F0..F4
PIXEL_NOISE = 20.0      # m/s per pixel per minute (Table III)

print(f'Wavelength scale: 1 m/s -> {1e3 * LAMBDA_NI / C_LIGHT * 1e8 / 1e3:.3f} micro-Angstrom')

## Part 1: Filter Cascade Transmission / 필터 캐스케이드 투과 프로파일

MDI's wavelength selection is a multi-stage cascade:
- 50 Å front window (broad — not modeled here)
- 8 Å blocker (broad — not modeled here)
- **465 mÅ Lyot** filter (Gaussian-like passband)
- **Michelson 1**: $T_{M1}(\lambda) = \tfrac{1}{2}[1+\cos(2\pi(\lambda-\lambda_1)/377\,\text{mÅ})]$
- **Michelson 2**: $T_{M2}(\lambda) = \tfrac{1}{2}[1+\cos(2\pi(\lambda-\lambda_2)/189\,\text{mÅ})]$

The product yields the 94 mÅ FWHM transmission peak that MDI tunes across the line.

Lyot + 두 Michelson을 곱하면 94 mÅ FWHM의 통과 피크가 생성됩니다.

In [ ]:
def lyot_transmission(lam_mAA: np.ndarray, center_mAA: float = 0.0,
                      fwhm_mAA: float = LYOT_FWHM) -> np.ndarray:
    """Approximate MDI Lyot filter transmission as a Gaussian.

    The actual Lyot is sech^2-like with a 4-2-2-4-6-8 cascade, but a Gaussian
    captures the dominant 465 mAA FWHM passband to within a few percent for our
    bookkeeping purposes.

    Args:
        lam_mAA: Wavelength offset from line center in milli-Angstroms.
        center_mAA: Lyot center wavelength offset in milli-Angstroms.
        fwhm_mAA: Full width at half maximum in milli-Angstroms.

    Returns:
        Transmission profile in [0, 1].
    """
    sigma = fwhm_mAA / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    return np.exp(-0.5 * ((lam_mAA - center_mAA) / sigma) ** 2)


def michelson_transmission(lam_mAA: np.ndarray, center_mAA: float,
                           fsr_mAA: float) -> np.ndarray:
    """Sinusoidal channel spectrum of an ideal polarizing Michelson.

    Args:
        lam_mAA: Wavelength offset from line center in milli-Angstroms.
        center_mAA: Tuning position (peak of the Michelson) in milli-Angstroms.
        fsr_mAA: Free spectral range in milli-Angstroms.

    Returns:
        Transmission profile in [0, 1].
    """
    return 0.5 * (1.0 + np.cos(2.0 * np.pi * (lam_mAA - center_mAA) / fsr_mAA))


def mdi_total_transmission(lam_mAA: np.ndarray, tuning_mAA: float,
                           lyot_center: float = 45.0) -> np.ndarray:
    """Combined Lyot + Michelson 1 + Michelson 2 transmission profile.

    Args:
        lam_mAA: Wavelength offsets in milli-Angstroms.
        tuning_mAA: Common tuning offset applied to both Michelsons.
        lyot_center: Lyot center wavelength offset in milli-Angstroms
            (Table II: ~45 mAA red of the mean solar wavelength).

    Returns:
        Total instrument transmission in [0, 1].
    """
    t_lyot = lyot_transmission(lam_mAA, center_mAA=lyot_center)
    t_m1 = michelson_transmission(lam_mAA, center_mAA=tuning_mAA, fsr_mAA=MICHELSON_M1_FSR)
    t_m2 = michelson_transmission(lam_mAA, center_mAA=tuning_mAA, fsr_mAA=MICHELSON_M2_FSR)
    return t_lyot * t_m1 * t_m2


lam = np.linspace(-2000, 2000, 8001)  # mAA grid
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(lam, lyot_transmission(lam), label='Lyot (465 mAA)', color='C0')
axes[0].plot(lam, michelson_transmission(lam, 0.0, MICHELSON_M1_FSR),
             label='Michelson 1 (FSR=377 mAA)', color='C1', alpha=0.7)
axes[0].plot(lam, michelson_transmission(lam, 0.0, MICHELSON_M2_FSR),
             label='Michelson 2 (FSR=189 mAA)', color='C2', alpha=0.7)
axes[0].set_ylabel('Transmission')
axes[0].set_title('Individual filter elements / 개별 필터 요소')
axes[0].legend(loc='upper right')
axes[0].grid(alpha=0.3)

for tuning, color in zip([-150, -75, 0, 75, 150], ['C3', 'C4', 'C5', 'C6', 'C7']):
    axes[1].plot(lam, mdi_total_transmission(lam, tuning_mAA=tuning),
                 label=f'F at {tuning:+d} mAA', color=color)
axes[1].set_ylabel('Transmission')
axes[1].set_title('Five MDI filtergram tunings (cascaded) / 5개 MDI 필터그램 튜닝 (캐스케이드)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(alpha=0.3)

# Verify FWHM of central tuning: should be ~94 mAA
t_total = mdi_total_transmission(lam, tuning_mAA=0.0)
half_max = t_total.max() / 2.0
above = lam[t_total >= half_max]
fwhm_measured = above.max() - above.min()
axes[2].plot(lam, t_total, color='black', lw=1.2)
axes[2].axhline(half_max, color='red', ls='--', alpha=0.5, label=f'Half max ({half_max:.3f})')
axes[2].axvspan(above.min(), above.max(), alpha=0.15, color='red',
                label=f'FWHM ~ {fwhm_measured:.0f} mAA (paper: 94 mAA)')
axes[2].set_xlabel('Wavelength offset from line center [mAA]')
axes[2].set_ylabel('Transmission')
axes[2].set_title('Cascaded passband and FWHM verification / 캐스케이드 통과대 및 FWHM 검증')
axes[2].legend(loc='upper right')
axes[2].grid(alpha=0.3)
axes[2].set_xlim(-500, 500)

plt.tight_layout()
plt.show()
print(f'Measured FWHM of cascaded transmission: {fwhm_measured:.1f} mAA (target: 94 mAA)')

## Part 2: MDI Doppler Velocity Proxy / MDI 도플러 속도 프록시

MDI computes Doppler velocity from five filtergrams $F_0, F_1, F_2, F_3, F_4$ at $\lambda_0, \lambda_0 \pm 75, \lambda_0 \pm 150$ mÅ via:

$$\alpha = \begin{cases} (F_1+F_2-F_3-F_4)/(F_1-F_3), & \text{if num} > 0 \\ (F_1+F_2-F_3-F_4)/(F_4-F_2), & \text{otherwise} \end{cases}$$

We simulate the Ni I 6768 line as a Gaussian absorption profile with FWHM 100 mÅ and depth 0.5, Doppler-shifted by some velocity, and integrate against the MDI transmission profiles to obtain $\alpha$ as a function of true velocity. This reproduces Fig. 12 (top panel).

5개 필터그램 $F_0, F_1, F_2, F_3, F_4$ ($\lambda_0, \lambda_0 \pm 75, \lambda_0 \pm 150$ mÅ)로부터 $\alpha$를 계산하여 도플러 속도를 lookup table로 변환합니다. 이 계산은 논문 Fig. 12의 상단 패널을 재현합니다.

In [ ]:
def solar_line_profile(lam_mAA: np.ndarray, velocity_ms: float,
                       depth: float = 0.5, fwhm_mAA: float = 100.0) -> np.ndarray:
    """Gaussian absorption line profile shifted by Doppler velocity.

    Args:
        lam_mAA: Wavelength offset from rest line center in milli-Angstroms.
        velocity_ms: Line-of-sight velocity in m/s (positive = redshift).
        depth: Fractional line depth (continuum = 1, line center = 1 - depth).
        fwhm_mAA: Line FWHM in milli-Angstroms (Ni I 6768 has ~100 mAA).

    Returns:
        Normalized intensity profile (continuum = 1.0).
    """
    sigma = fwhm_mAA / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    # Doppler shift in mAA: dlambda = lambda * v / c
    shift_mAA = LAMBDA_NI * velocity_ms / C_LIGHT * 1e3  # mAA
    return 1.0 - depth * np.exp(-0.5 * ((lam_mAA - shift_mAA) / sigma) ** 2)


def filtergram_intensities(velocity_ms: float, lam_grid: np.ndarray,
                           tunings_mAA=(0.0, 75.0, 150.0, -75.0, -150.0),
                           **line_kwargs) -> np.ndarray:
    """Compute the five MDI filtergram intensities F0..F4 at a given velocity.

    Note that MDI labels F0=center, F1=+75, F2=+150, F3=-75, F4=-150 mAA;
    here we use that ordering.

    Args:
        velocity_ms: Line-of-sight velocity in m/s.
        lam_grid: Wavelength grid for integration (milli-Angstroms).
        tunings_mAA: Tuple of tuning offsets for F0..F4.

    Returns:
        Array of five filtergram intensities (F0, F1, F2, F3, F4).
    """
    line = solar_line_profile(lam_grid, velocity_ms, **line_kwargs)
    intensities = np.zeros(5)
    for i, tune in enumerate(tunings_mAA):
        t_filter = mdi_total_transmission(lam_grid, tuning_mAA=tune)
        intensities[i] = np.trapz(line * t_filter, lam_grid)
    return intensities


def alpha_proxy(F: np.ndarray) -> float:
    """MDI Doppler proxy alpha from filtergram intensities F=(F0,F1,F2,F3,F4).

    Implements the piecewise definition from Scherrer et al. 1995, Sec. 4.1.

    Args:
        F: Array of five filtergram intensities in MDI ordering.

    Returns:
        Dimensionless Doppler proxy alpha.
    """
    F0, F1, F2, F3, F4 = F
    num = F1 + F2 - F3 - F4
    if num > 0:
        return num / (F1 - F3) if (F1 - F3) != 0 else 0.0
    return num / (F4 - F2) if (F4 - F2) != 0 else 0.0


# Generate alpha vs velocity calibration curve / 보정 곡선 생성
lam_int = np.linspace(-1500, 1500, 6001)
velocities = np.linspace(-4000, 4000, 161)  # m/s
alphas_nominal = np.array([alpha_proxy(filtergram_intensities(v, lam_int)) for v in velocities])
alphas_broad = np.array([alpha_proxy(filtergram_intensities(v, lam_int, fwhm_mAA=125.0)) for v in velocities])
alphas_narrow = np.array([alpha_proxy(filtergram_intensities(v, lam_int, fwhm_mAA=75.0)) for v in velocities])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(velocities, alphas_nominal, label='Nominal line (FWHM=100 mAA)', color='black', lw=2)
ax.plot(velocities, alphas_broad, label='25% broader (125 mAA)', color='C1', ls='--')
ax.plot(velocities, alphas_narrow, label='25% narrower (75 mAA)', color='C2', ls=':')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('True Doppler velocity [m/s]')
ax.set_ylabel(r'MDI Doppler proxy $\alpha$')
ax.set_title('MDI velocity calibration curve (reproduces Scherrer 1995 Fig. 12 top)\n'
             'MDI 속도 보정 곡선 (Scherrer 1995 Fig. 12 상단 재현)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim(-4000, 4000)
plt.tight_layout()
plt.show()

# Verify monotonicity / 단조성 검증
is_monotonic = np.all(np.diff(alphas_nominal) > 0)
print(f'alpha is monotonic in velocity: {is_monotonic}')
print(f'alpha range: [{alphas_nominal.min():.3f}, {alphas_nominal.max():.3f}]')

### Velocity error from line-width mismatch / 선폭 불일치로부터의 속도 오차

Inverting the nominal $\alpha(v)$ curve via interpolation gives a velocity lookup. Applying it to $\alpha$ values from broader/narrower lines yields the systematic velocity error shown in Fig. 12 (bottom panel).

공칭 $\alpha(v)$ 곡선을 보간으로 역변환하여 속도 lookup을 얻고, 이를 더 넓거나 좁은 선의 $\alpha$에 적용하면 Fig. 12 하단 패널의 계통오차를 얻습니다.

In [ ]:
def velocity_from_alpha(alpha_value: float, alpha_table: np.ndarray,
                        v_table: np.ndarray) -> float:
    """Invert alpha to velocity via the calibration lookup.

    Args:
        alpha_value: Measured proxy value.
        alpha_table: Tabulated alpha values (monotonic).
        v_table: Corresponding tabulated velocities in m/s.

    Returns:
        Inferred velocity in m/s.
    """
    return float(np.interp(alpha_value, alpha_table, v_table))


v_recovered_broad = np.array([velocity_from_alpha(a, alphas_nominal, velocities) for a in alphas_broad])
v_recovered_narrow = np.array([velocity_from_alpha(a, alphas_nominal, velocities) for a in alphas_narrow])

err_broad = v_recovered_broad - velocities
err_narrow = v_recovered_narrow - velocities

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(velocities, err_broad, label='Broader line (+25%)', color='C1', lw=1.5)
ax.plot(velocities, err_narrow, label='Narrower line (-25%)', color='C2', lw=1.5)
ax.axhline(0, color='black', lw=0.7)
ax.axhline(150, color='gray', ls='--', lw=0.7, label='Paper bound: +/-150 m/s')
ax.axhline(-150, color='gray', ls='--', lw=0.7)
ax.set_xlabel('True velocity [m/s]')
ax.set_ylabel('Velocity error: MDI - true [m/s]')
ax.set_title('Systematic velocity error from line-width mismatch (Scherrer 1995 Fig. 12 bottom)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(-4000, 4000)
plt.tight_layout()
plt.show()
print(f'Max error within +/-3000 m/s, broader line: {np.max(np.abs(err_broad[(velocities>=-3000)&(velocities<=3000)])):.0f} m/s')
print(f'Max error within +/-3000 m/s, narrower line: {np.max(np.abs(err_narrow[(velocities>=-3000)&(velocities<=3000)])):.0f} m/s')

## Part 3: Simulated $k$-$\omega$ p-mode Power Spectrum / 합성 $k$-$\omega$ p-mode 파워 스펙트럼

MDI's primary product is the $\ell$-$\nu$ diagram (Fig. 16 of the paper). For pedagogical purposes we simulate a 1-D analog: a $k$-$\omega$ diagram from a synthetic time series of solar oscillations on a strip of the photosphere.

The dispersion relation for solar p-modes near the surface is approximately:

$$\omega^2 \approx c_s^2 k^2 + \omega_c^2$$

where $c_s \sim 7$ km/s is the sound speed and $\omega_c \sim 5.3$ mHz is the acoustic cutoff. Trapped modes form discrete ridges below the cutoff. We synthesize a velocity field $v(x, t) = \sum_{n,k} A_{n,k} \cos(kx - \omega_{n,k} t + \phi_{n,k})$ and compute its 2-D power spectrum.

MDI의 주요 산출물은 $\ell$-$\nu$ 도표 (논문 Fig. 16)입니다. 교육 목적으로 광구의 한 줄을 따라 합성된 진동 시계열로부터 1-D 유사물인 $k$-$\omega$ 도표를 시뮬레이션합니다.

In [ ]:
def simulate_pmode_strip(n_x: int = 256, n_t: int = 512, dx_km: float = 1500.0,
                         dt_s: float = 60.0, n_modes: int = 200,
                         seed: int = 42) -> tuple:
    """Synthesize a 1-D photospheric Doppler time series with p-mode ridges.

    The model uses a discrete spectrum of trapped modes whose dispersion
    follows a Lamb-like relation omega^2 = (c_s k)^2 + omega_c^2 plus discrete
    radial ridges at omega_n = sqrt((c_s k)^2 + (n * pi * c_s / H)^2).

    Args:
        n_x: Number of spatial samples along the strip.
        n_t: Number of time samples.
        dx_km: Spatial sampling in km (MDI 4" ~ 2900 km).
        dt_s: Temporal sampling in seconds (MDI cadence: 60 s).
        n_modes: Number of randomly drawn modes to superpose.
        seed: RNG seed.

    Returns:
        (v_xt, x_grid_km, t_grid_s) -- velocity time series and axes.
    """
    rng = np.random.default_rng(seed)
    x = np.arange(n_x) * dx_km            # km
    t = np.arange(n_t) * dt_s             # s
    X, T = np.meshgrid(x, t, indexing='ij')

    c_s = 7.0   # km/s sound speed
    H = 200.0   # km approximate scale height producing ridge spacing
    omega_c_mHz = 5.3
    omega_c = 2.0 * np.pi * omega_c_mHz * 1e-3  # rad/s

    v_xt = np.zeros((n_x, n_t))
    for _ in range(n_modes):
        n_radial = rng.integers(1, 8)
        k_per_km = rng.uniform(2 * np.pi / (n_x * dx_km) * 4,
                               2 * np.pi / (2.0 * dx_km))  # 1/km
        omega_n = np.sqrt((c_s * k_per_km) ** 2 + (n_radial * np.pi * c_s / H) ** 2 * 1e-2)
        # restrict to omega < omega_cutoff to mimic trapping
        if omega_n * 1e-3 > omega_c * 2:  # too high
            continue
        amp = rng.uniform(0.3, 1.0)
        phase = rng.uniform(0, 2 * np.pi)
        v_xt += amp * np.cos(k_per_km * X * 1e3 - omega_n * 1e-3 * T + phase)

    # Add white noise comparable to MDI per-pixel-per-minute level / MDI 노이즈 추가
    v_xt += rng.normal(scale=PIXEL_NOISE / 1000.0, size=v_xt.shape)
    return v_xt, x, t


v_xt, x_km, t_s = simulate_pmode_strip()
print(f'Simulated strip: {v_xt.shape[0]} pixels x {v_xt.shape[1]} time samples')
print(f'Spatial extent: {x_km.max():.0f} km;  duration: {t_s.max() / 60:.0f} min')

In [ ]:
def kw_power_spectrum(v_xt: np.ndarray, dx_km: float, dt_s: float) -> tuple:
    """Compute 2-D k-omega power spectrum of a (x, t) Doppler strip.

    Args:
        v_xt: Velocity array of shape (n_x, n_t).
        dx_km: Spatial sampling in km.
        dt_s: Temporal sampling in seconds.

    Returns:
        (power, k_axis_per_Mm, freq_axis_mHz) -- 2-D power and 1-D axes.
    """
    n_x, n_t = v_xt.shape
    # 2D FFT
    V = np.fft.fft2(v_xt - v_xt.mean())
    power = np.abs(V) ** 2
    # axes
    k_per_km = np.fft.fftfreq(n_x, d=dx_km) * 2.0 * np.pi  # rad/km
    freq_Hz = np.fft.fftfreq(n_t, d=dt_s)
    # Keep positive halves
    n_k_pos = n_x // 2
    n_f_pos = n_t // 2
    power_pos = power[:n_k_pos, :n_f_pos]
    return (power_pos,
            k_per_km[:n_k_pos] * 1e3,        # rad/Mm for plotting
            freq_Hz[:n_f_pos] * 1e3)         # mHz


power, k_axis, freq_axis = kw_power_spectrum(v_xt, dx_km=1500.0, dt_s=60.0)

fig, ax = plt.subplots(figsize=(10, 7))
log_power = np.log10(power + 1e-12)
im = ax.pcolormesh(k_axis, freq_axis, log_power.T, shading='auto',
                   cmap='inferno', vmin=np.percentile(log_power, 50),
                   vmax=np.percentile(log_power, 99))
ax.set_xlabel('Wavenumber k [rad/Mm]')
ax.set_ylabel('Frequency [mHz]')
ax.set_title('Simulated $k$-$\\omega$ p-mode power spectrum (analog of MDI Fig. 16)\n'
             '합성 $k$-$\\omega$ p-mode 파워 스펙트럼 (MDI Fig. 16의 아날로그)')
ax.set_ylim(0, 8)
ax.set_xlim(0, k_axis.max())
fig.colorbar(im, ax=ax, label='log10(power)')

# Overlay theoretical Lamb dispersion / 이론 분산 곡선 표시
k_overlay = np.linspace(0.1, k_axis.max(), 200)
c_s_Mm_per_s = 7.0e-3  # 7 km/s = 7e-3 Mm/s
omega_c_mHz_theory = 5.3
omega_lamb = np.sqrt((c_s_Mm_per_s * k_overlay) ** 2 +
                     (omega_c_mHz_theory * 1e-3 * 2 * np.pi) ** 2)
ax.plot(k_overlay, omega_lamb / (2 * np.pi) * 1e3,
        color='cyan', ls='--', lw=1.5, alpha=0.7,
        label='Lamb cutoff: $\\omega^2 = (c_s k)^2 + \\omega_c^2$')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | This paper / 이 논문 | Modern equivalent / 현대 동등물 |
|---|---|---|
| Spectral line / 흡수선 | Ni I 6767.78 Å (MDI) | Fe I 6173.34 Å (HMI on SDO) |
| Filter cascade / 필터 캐스케이드 | 50 Å + 8 Å + 465 mÅ Lyot + 188 + 94 mÅ Michelsons | Same architecture in HMI; Lyot retired in some designs |
| Doppler proxy / 도플러 프록시 | 5-filtergram piecewise $\alpha$ ratio | HMI uses 6 filtergrams; Gaussian/Voigt fit as alternative |
| Spatial resolution / 공간 분해능 | 4″ FD, 1.25″ HR | 1″ HMI |
| Cadence / 카덴스 | 60 s | 45 s (HMI) |
| Detector / 검출기 | 1024² CCD (Loral) | 4096² CCD (HMI) |
| Velocity noise / 속도 노이즈 | 20 m/s/pix/min | ~10 m/s/pix/min (HMI) |
| Power spectrum / 파워 스펙트럼 | $\ell$-$\nu$ via spherical harmonic decomposition | Same; modes $\ell \le 6000$ |
| Pipeline / 파이프라인 | SSSC (Stanford), Levels 0-3 | JSOC; same level structure |

### Key takeaways from the implementation / 구현으로부터의 핵심 시사점

1. The 94 mÅ FWHM of MDI emerges naturally from the product of a wide Lyot ($\sim$465 mÅ) with two sinusoidal Michelsons (FSR 377 and 189 mÅ). The narrower Michelson (M2) sets the final FWHM; the broader (M1) suppresses adjacent transmission peaks of M2; the Lyot suppresses sidelobes of M1.

2. The piecewise $\alpha$ proxy has a **monotonic, sigmoid-like** response over $\pm 4000$ m/s. Inverting it via a precomputed lookup table gives Doppler velocity insensitive to gain and offset.

3. Line-width mismatch produces systematic velocity errors of $\lesssim 150$ m/s for $\pm 25\%$ width changes within $\pm 3000$ m/s, consistent with Fig. 12 of the paper.

4. A simulated $k$-$\omega$ power spectrum from a 1-D photospheric strip with 60-s cadence and 4″-class spatial sampling reproduces the qualitative ridge structure of the MDI $\ell$-$\nu$ diagram (Fig. 16), illustrating why long uninterrupted time series are essential for high-precision mode frequency extraction.